In [1]:
import numpy as np
import pandas as pd
import bioframe
import cooltools
import cooler
import matplotlib.pyplot as plt
import seaborn as sns
from sample import *
from supp_lib import *
# make matplotlib pdf-s text recognizable by evil-Adobe
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

In [ ]:
c = cooler.Cooler("hg38/OU-HiC-N6IPSC2-MN-W6__hg38.hg38.mapq_30.1000.mcool::/resolutions/250000")

In [ ]:
chr1:900000-248956422
chr2:1-242193529
chr3:1-198295559
chr4:100000-190214555
chr5:1-181538259
chr6:100000-170805979
chr7:1-159345973
chr8:200000-145138636
chr9:200000-138394717
chr10:100000-133797422
chr11:300000-135086622
chr12:1-133275309
chr13:19100000-114364328
chr14:20200001-107043718
chr15:22800000-101991189
chr16:1-90338345
chr17:100000-83257441
chr18:100000-80373285
chr19:300000-58617616
chr20:100000-64444167
chr21:14000000-46709983
chr22:20000000-50818468

In [2]:
regs_w=['chr1','chr2','chr3','chr4','chr5','chr6','chr7','chr8',
     'chr9','chr10','chr11','chr12','chr13','chr14','chr15','chr16',
     'chr17','chr18','chr19','chr20','chr21','chr22']

In [ ]:
regs=['chr1:900000-248956422','chr2:1-242193529','chr3:1-198295559','chr4:100000-190214555',
'chr5:1-181538259','chr6:100000-170805979','chr7:1-159345973','chr8:200000-145138636',
'chr9:200000-138394717','chr10:100000-133797422','chr11:300000-135086622','chr12:1-133275309',
'chr13:19100000-114364328','chr14:20200001-107043718','chr15:22800000-101991189','chr16:1-90338345',
'chr17:100000-83257441','chr18:100000-80373285','chr19:300000-58617616','chr20:100000-64444167',
'chr21:14000000-46709983','chr22:20000000-50818468']

In [3]:
def generate_matrix(regs_w,c):
    regs=regs_w
    print(len(regs))
    #set ticks alternating
    ticks = [''] * len(regs)*2
    for i in range(0,(len(regs)*2),2):
        ticks[i]=regs[int(i/2)]
    #print(ticks)

    #print(len(regs)*2)
    #setting ,mask for cells to annotate
    y = np.ones((len(regs)*2,len(regs)*2)) 
    ticks = [''] * len(regs)*2
    annot_mask=np.zeros((len(regs)*2,len(regs)*2)) 

    for i in range(0,len(regs)*2,2):
        ticks[i]=regs[int(i/2)]
        for j in range(i,len(regs)*2,2):
            #print(i,j)
            r1=regs[int(i/2)]
            r2=regs[int(j/2)]
            #print(r1,"-",r2,end=",")

            if(i==j):
                y[i,j]=y[i,j+1]=y[i+1,j]=y[i+1,j+1]=np.nansum(c.matrix(balance=True).fetch(r1, r2)[:4,-4:])
                #print(y[i,j])
            else:
                mat1=np.nansum(c.matrix(balance=True).fetch(r1, r2)[:4,:4])
                mat2=np.nansum(c.matrix(balance=True).fetch(r1, r2)[:4,-4:])
                mat3=np.nansum(c.matrix(balance=True).fetch(r1, r2)[-4:,:4])
                mat4=np.nansum(c.matrix(balance=True).fetch(r1, r2)[-4:,-4:])
                #print(np.sum(np.isnan(mat1)),np.sum(np.isnan(mat2)),np.sum(np.isnan(mat3)),np.sum(np.isnan(mat4)))
                #print(np.nansum(mat1),np.nansum(mat2),np.nansum(mat3),np.nansum(mat4))
                y[i,j]=mat1
                y[i,j+1]=mat2
                y[i+1,j]=mat3
                y[i+1,j+1]=mat4
                #print(mat1,mat2,mat3,mat4)
                annot_mask[i,j]=annot_mask[i,j+1]=annot_mask[i+1,j]=annot_mask[i+1,j+1]=1
            '''
            s=np.nansum(mat1)+np.nansum(mat2)+np.nansum(mat3)+np.nansum(mat4)
            print(s)
            y[i,j]=s
            '''
    return y,ticks,annot_mask

ys = np.empty((2, 2)) 
#regs=['chr9','chr10']
r1='chr1'
r2='chr2'

ys[0,0]=np.nansum(c.matrix(balance=True).fetch(r1, r2)[:4,:4])
ys[0,1]=np.nansum(c.matrix(balance=True).fetch(r1, r2)[:4,-4:])
ys[1,0]=np.nansum(c.matrix(balance=True).fetch(r1, r2)[-4:,:4])
ys[1,1]=np.nansum(c.matrix(balance=True).fetch(r1, r2)[-4:,-4:])
#print(np.sum(np.isnan(mat1)),np.sum(np.isnan(mat2)),np.sum(np.isnan(mat3)),np.sum(np.isnan(mat4)))
#print(np.nansum(mat1),np.nansum(mat2),np.nansum(mat3),np.nansum(mat4))
#s=np.nansum(mat1)+np.nansum(mat2)+np.nansum(mat3)+np.nansum(mat4)
#print(s)
ys

In [4]:
def plot_matrix(y,ticks,annot_mask,n):
    mask = (y == 1)
    plt.figure(figsize=(40, 40))
    sns.heatmap(y, annot=False, cmap='YlOrBr', mask=mask, vmin=0, vmax=0.06)

    for i in range(len(ticks)):
        for j in range(len(ticks)):
            if(annot_mask[i,j]==1):
                plt.text(j + 0.5, i + 0.5, f'{y[i, j]:.4f}', ha='center', va='center', color='black',rotation=45,fontsize=8)
        
    for v in range(0,len(ticks),2):
        #print(v,v)
        plt.text(v + 1, v + 1, f'{y[v, v]:.4f}', ha='center', va='center', color='black',fontsize=8)
    
    plt.title(n)

    plt.xticks(ticks=[i + 1 for i in range(len(ticks))], labels=ticks, rotation=0)
    plt.yticks(ticks=[i + 1 for i in range(len(ticks))], labels=ticks, rotation=0)


    plt.tick_params(axis='x', which='both', bottom=False, top=True, labelbottom=False, labeltop=True)
    plt.tick_params(axis='y', which='both', left=False, right=True, labelleft=False, labelright=True)

    for i in range(0, len(ticks), 2):
        plt.plot((i, i), (0, i+2), 'k-')
        plt.plot((i, len(ticks)), (i+2, i+2), 'k-')
        #plt.axvline(x=i,ymin=i/len(ticks),ymax=6,color='black', linewidth=1)

    plt.plot((0, len(ticks)), (0, 0), 'k-',linewidth=3)
    plt.plot((len(ticks), len(ticks)), (0, len(ticks)), 'k-',linewidth=3)
    plt.savefig("cooltools54/Telomere_Interactions/"+n+".pdf")
    #plt.show()


In [ ]:
for f in (indi+grps):
    print(f)
    c = cooler.Cooler("hg38/"+f+"__hg38.hg38.mapq_30.1000.mcool::/resolutions/250000")
    y,ticks,annot_mask=generate_matrix(regs_w,c)
    plot_matrix(y,ticks,annot_mask,f)